# Lab - Conversation State in Customer Sessions

Three tasks on where a conversation's memory lives. The Claude API remembers nothing between requests, and a support chat is a chain of requests - from many customers at once.

| Task | What you learn |
| --- | --- |
| 1 | The API is stateless: a request knows only what it carries |
| 2 | One history list per customer session, sent in full every turn |
| 3 | Sessions never mix, and each one pays for its own growing history |

**You do not need an API key.** This lab ships with a Claude simulator, so every cell
runs offline. The code you write is exactly the code you would write against the real
API - set `ANTHROPIC_API_KEY` at home and the same notebook calls Claude for real.

**You do not write code from scratch.** Each cell already holds the code, with blanks
marked `...` and a comment telling you what goes in each one.

## Setup

Run this cell first, before anything else. Click it, then press **Shift+Enter**.

In [ ]:
# --- Lab setup (provided - just run it) ---
import shopassist_lab
from shopassist_lab import check

# Everything below this line is ordinary Claude API code.
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()
client = Anthropic()

# The model name lives in one constant, so a later update changes one line.
MODEL = "claude-sonnet-5-5"
SYSTEM = (
    "You are ShopAssist, an online store's support assistant. "
    "Returns are accepted within 30 days of delivery. "
    "Answer in one or two sentences."
)

---

## Task 1 - The API keeps no state

In [ ]:
# ============================================================
# TASK 1 - The API keeps no state
# ============================================================
#
# WHAT TO DO
#   Finish ask(): it sends a list of messages with the system
#   line and returns the reply text and the response. Then send
#   the customer's two messages as two separate requests.
#
# WHY IT MATTERS
#   The second answer will ask which item you mean. It is
#   tempting to think Claude forgot. Nothing was forgotten,
#   because nothing was ever kept: the API is stateless, and
#   Claude sees only what one request contains. For a developer
#   that settles one thing - conversation state is your
#   application's job.
#
# WHERE TO SEE IT IN THE LECTURE
#   "The API is stateless".
#
# HOW TO DO IT
#   Three blanks:
#
#       SYSTEM     the system line from the setup cell
#       messages   the list ask() received
#       "text"     the type of block that holds the reply
# ============================================================

# TODO: replace each ... below with the value named beside it
def ask(messages):
    response = client.messages.create(
        model=MODEL,
        max_tokens=200,
        output_config={"effort": "low"},
        system=...,     # SYSTEM
        messages=...,   # messages
    )
    reply = None
    for block in response.content:
        if block.type == ...:   # "text"
            reply = block.text
    return reply, response

second = [{"role": "user", "content": "Can I still return it?"}]   # provided
first_reply, _ = ask([{"role": "user", "content": "I bought a jacket 20 days ago."}])
second_reply, _ = ask(second)
print("1:", first_reply)
print("2:", second_reply)

check("no_state", first_reply=first_reply, second_reply=second_reply, sent=len(second))

---

## Task 2 - One history per customer session

In [ ]:
# ============================================================
# TASK 2 - One history per customer session
# ============================================================
#
# WHAT TO DO
#   Keep a history list for every customer, in a dictionary
#   keyed by session id. say() adds the customer's message to
#   that session's list, sends the whole list, and adds
#   Claude's reply as an assistant message.
#
# WHY IT MATTERS
#   The memory of a conversation lives in your code: a list,
#   sent in full with every request. A real application talks
#   to many customers at once, so there is not one list but one
#   per session - and the session id decides which list a
#   message belongs to.
#
# WHERE TO SEE IT IN THE LECTURE
#   "So the memory of the conversation must live in our code".
#
# HOW TO DO IT
#   Three blanks:
#
#       session_id    the key of this customer's list
#       "assistant"   the role of Claude's reply in the history
#       history       the whole list, sent every turn
# ============================================================

SESSIONS = {}   # provided: session id -> that customer's history
TOKENS = {}     # provided: session id -> input tokens of each turn

# TODO: replace each ... below with the value named beside it
def say(session_id, text):
    history = SESSIONS.setdefault(..., [])   # session_id
    history.append({"role": "user", "content": text})
    reply, response = ask(...)   # history - the whole list
    history.append({"role": ..., "content": reply})   # "assistant"
    TOKENS.setdefault(session_id, []).append(response.usage.input_tokens)
    print(f"[{session_id}] {reply}")

SESSIONS.clear(); TOKENS.clear()   # provided: re-running this cell starts fresh
try:
    say("alex", "I bought a jacket 20 days ago.")
    say("alex", "Can I still return it?")
    say("alex", "And if I lost the receipt?")
except (TypeError, KeyError, AttributeError) as exc:   # provided: a blank left above, said plainly
    if "llipsis" in str(exc):
        raise ValueError("You still have ... above. Replace each one with the value "
                         "named beside it.") from None
    raise

check("session_history", sessions=SESSIONS, say=say)

---

## Task 3 - Sessions never mix

In [ ]:
# ============================================================
# TASK 3 - Sessions never mix
# ============================================================
#
# WHAT TO DO
#   A second customer, Sam, opens a chat while Alex's is still
#   going and asks the same question. Send Sam's message into
#   Sam's own session, then compare what each session costs.
#
# WHY IT MATTERS
#   One shared history would hand Sam Alex's jacket - and Alex's
#   personal details. Separate sessions keep each customer's
#   state apart. And each session pays for its own memory: the
#   input tokens grow with every turn, because every turn sends
#   the whole history again. A long chat can reach the context
#   window - the most one request can hold.
#
# WHERE TO SEE IT IN THE LECTURE
#   "Look at the input tokens".
#
# HOW TO DO IT
#   Three blanks:
#
#       "sam"                      Sam's own session id
#       "Can I still return it?"   Sam's question
#       TOKENS                     the input tokens, per session
# ============================================================

try:
    # TODO: replace each ... below with the value named beside it
    say(..., ...)   # "sam", "Can I still return it?"
    for session_id, counts in ....items():   # TOKENS
        print(session_id, "input tokens per turn:", counts)
except (TypeError, KeyError, AttributeError) as exc:   # provided: a blank left above, said plainly
    if "llipsis" in str(exc):
        raise ValueError("You still have ... above. Replace each one with the value "
                         "named beside it.") from None
    raise

check("session_isolation", sessions=SESSIONS, tokens=TOKENS)

---

## Done

The API keeps no state, so your application does: one history per session, sent in full with every request. Separate sessions keep customers apart, and every session's input tokens grow with its history.